# Item 97: Rely on Precompiled Bytecode and File System Caching to

Improve Startup Times

## Notes

-   Program start-up time is a key performance metric
    -   Directly observable by the user
-   Programs that are constantly rerun accumulate delay from the
    start-up time e.g.
    -   Commonly used command line utilities
    -   Web servers starting up request processing
-   Python has a high start-up time (at least in CPython (See [Item
    1](../../Chapter_01/Item_001/item_001.qmd)))
    -   Source code is read and compiled into bytecode
        -   Uses CPU time and I/O (See [Item
            68](../../Chapter_09/Item_068/item_068.qmd))
    -   Once bytecode is ready, all imported modules are executed
        -   Requires running code to,
            -   Initialise global variables and constants
            -   Define classes and functions
            -   Execute assert statements etc.
-   Modules are cached in memory after the first load
    -   Subsequent loads will reuse them
-   CPython saves generated bytecode to disk
    -   Can be reused by subsequent invocations
    -   Stored in `__pycache__` directory at same level as source code
    -   Typically suffixed as `.pyc`
-   Caching can provide some speed-up, e.g. loading all django framework
    modules might get numbers like below

``` shell
$ time python3 -c 'import django_all'

real    0m0.791s
user    0m0.495s
sys     0m0.145s
```

-   Then running again with cached bytecode

``` shell
$ time python3 -c 'import django_all'

real    0m0.225s
user    0m0.182s
sys     0m0.038s
```

-   However if we also remove the bytecode cache and then run the
    program again we might still see a speed-up over the initial run

``` shell
$ time python3 -c 'import django_all'

real    0m0.613s
user    0m0.502s
sys     0m0.101s
```

-   This is because rerunning the program at the operating system level
    is mixing optimisations
    -   In the above example even though python has to recompile the
        code the operating system filesystem has *cached* access to the
        files
-   Bytecode can be regenerated by the `compileall` built-in module
    -   Normally done automatically
    -   Can create it manually if needed

``` shell
$ python3 -m compileall django

Listing 'django'
Compiling 'django/__init__.py'
Compiling 'django/__main__.py'
Listing 'django/apps'
Compiling 'django/apps/__init__.py'
Compiling 'django/apps/config.py'
Compiling 'django/apps/registry.py'
...
```

-   Operating systems normally also provide a mechanism to purge the
    system file cache
    -   Let’s us time the increase from generating the bytecode without
        the file system caching influence

``` shell
$ sudo purge
$ time python3 -c 'import django_all'

real    0m0.382s
user    0m0.169s
sys     0m0.085s
```

-   So we can see that running from compiled bytecode with a cold cache
    is faster than running with no bytecode and a hot cache
-   Still slower than running from a hot cache
    -   An option if start-up performance is critical is to set-up
        python such that is is always in memory, e.g. RAM disk
-   When caching is not possible consider other avenues (See [Item
    98](../Item_098/item_098.qmd))
-   To run a python program purely off compiled bytecode use the `-b`
    flag
    -   `.pyc` files are put next to the source rather than in
        `__pycache__`
    -   But there is no real speed-up here (Here working with a hot
        cache)

``` shell
$ find django -name '*.pyc' -delete
$ python3 -m compileall -b django
$ find django -name '*.py' -delete
$ time python3 -c 'import django_all'

real    0m0.226s
user    0m0.183s
sys     0m0.037s
```

## Things to Remember

-   CPython compiles python programs into bytecode at start-up
    -   This bytecode is executed by the python virtual machine
-   Bytecode is cached to disk
    -   Subsequent runs of a program or module loads can avoid
        recompiling the bytecode
-   Best performance is achieved when the bytecode is pre-compiled and
    cached in memory